# Урок 5. Иерархические модели и деревья

9 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 4](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-04.ipynb) · [Урок 6 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-06.ipynb)

---

Дерево как частный случай графа. Корень, узлы, листья, уровни. Файловая система и классификации как деревья.

In [ ]:
#@title 🚀 Шаг 1. Регистрация и подготовка урока { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
ФИО = "" #@param {type:"string"}
Класс = "9А" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-05", name=ФИО, klass=Класс)

## Разбираемся

### Особый вид графа

Среди графов есть один, который встречается чаще всех остальных вместе
взятых, — **дерево**.

> **Дерево** — связный граф без циклов.

Определение короткое, но из него следует всё остальное. Раз циклов нет,
то между любыми двумя вершинами существует **ровно один** путь.
Не «хотя бы один» и не «много», а именно один — это главное свойство
дерева и причина его популярности.

### Словарь терминов

Деревья рисуют «вверх ногами»: корень сверху, листья снизу.

```
                 Школа              ← корень (уровень 0)
                ╱     ╲
         Началка       Средняя      ← уровень 1
          ╱   ╲         ╱    ╲
         1а   1б      5а     5б     ← уровень 2, листья
```

| Термин | Что означает |
|---|---|
| **корень** | единственная вершина без родителя |
| **потомок** | вершина уровнем ниже, связанная с данной |
| **родитель** | вершина уровнем выше |
| **лист** | вершина без потомков |
| **уровень** | расстояние от корня, у корня 0 |
| **высота** | наибольший уровень в дереве |
| **поддерево** | вершина вместе со всеми её потомками |

### Где вы встречаете деревья каждый день

**Файловая система.** Папки содержат папки и файлы, у каждого элемента
ровно один родитель, циклов нет. Путь `C:/Users/Аня/фото.jpg` —
это описание пути от корня к листу.

**Классификации.** Царство → тип → класс → отряд → семейство → род → вид.
Каждый организм попадает в единственную ветвь.

**Оглавление книги.** Части, главы, параграфы.

**Турнирная сетка.** Каждый матч — вершина, победители поднимаются
к корню-финалу.

**Дерево решений.** Каждая развилка — вопрос, каждый лист — вывод.
Так работают многие алгоритмы в медицине и технике.

### Сколько вершин на каждом уровне

В **полном двоичном дереве** у каждой невисячей вершины ровно два потомка.
Тогда количество вершин на уровне растёт как степень двойки:

| Уровень | 0 | 1 | 2 | 3 | 4 | ... | k |
|---|---|---|---|---|---|---|---|
| Вершин | 1 | 2 | 4 | 8 | 16 | ... | 2ᵏ |

А всего вершин в дереве высоты h будет $2^{h+1} - 1$.

Отсюда важный вывод: **дерево растёт вширь очень быстро**. Двадцать
уровней дают больше миллиона листьев. Именно поэтому поиск по дереву
так эффективен: чтобы найти нужный элемент среди миллиона, достаточно
двадцати шагов.

Эту же идею вы уже встречали — в формуле $N = 2^i$ из 7 класса.
Двадцать вопросов «да/нет» различают миллион вариантов. Дерево
и есть наглядное изображение этих вопросов.

### Как задать дерево в программе

Проще всего — словарём «родитель → список потомков», как обычный граф:

```python
дерево = {
    "Школа": ["Началка", "Средняя"],
    "Началка": ["1а", "1б"],
    "Средняя": ["5а", "5б"],
    "1а": [], "1б": [], "5а": [], "5б": [],
}
```

Второй способ — наоборот, «потомок → родитель». Он компактнее и удобен,
когда нужно подниматься к корню:

```python
родители = {"Началка": "Школа", "1а": "Началка", ...}
```

## Смотрим, как это работает

### Пример 1. Листья и уровни

In [ ]:
дерево = {
    "Школа": ["Началка", "Средняя"],
    "Началка": ["1а", "1б"],
    "Средняя": ["5а", "5б", "9а"],
    "1а": [], "1б": [], "5а": [], "5б": [], "9а": [],
}

листья = [в for в, потомки in дерево.items() if not потомки]
print("Листья:", листья)
print("Всего вершин:", len(дерево))
print("Внутренних вершин:", len(дерево) - len(листья))

Лист определяется одним признаком — пустым списком потомков.
Проверка `if not потомки` истинна для пустого списка,
что читается почти как обычная фраза.

### Пример 2. Обход дерева и подсчёт уровней

Пройдём по дереву от корня, отмечая уровень каждой вершины.

In [ ]:
def уровни(дерево, корень):
    результат = {}
    очередь = [(корень, 0)]

    while очередь:
        вершина, уровень = очередь.pop(0)
        результат[вершина] = уровень
        for потомок in дерево[вершина]:
            очередь.append((потомок, уровень + 1))

    return результат


глубины = уровни(дерево, "Школа")
for вершина, уровень in глубины.items():
    print(f"  {'    ' * уровень}{вершина}  (уровень {уровень})")

print(f"\nВысота дерева: {max(глубины.values())}")

Здесь использован **обход в ширину**: мы берём вершины из очереди
по одной, а её потомков добавляем в конец. Поэтому дерево обходится
уровень за уровнем — сначала все вершины уровня 1, потом уровня 2.

Метод `pop(0)` забирает элемент с начала списка, что и делает список
очередью. Отступ в печати `'    ' * уровень` наглядно показывает
структуру — тот же приём, что в оглавлениях книг.

### Пример 3. Путь от вершины к корню

Задача, которая постоянно возникает с файловыми системами:
восстановить полный путь.

In [ ]:
родители = {
    "Началка": "Школа",
    "Средняя": "Школа",
    "1а": "Началка",
    "1б": "Началка",
    "5а": "Средняя",
    "9а": "Средняя",
}


def путь_к_корню(родители, вершина):
    путь = [вершина]
    while вершина in родители:
        вершина = родители[вершина]
        путь.append(вершина)
    return путь[::-1]


for вершина in ["1а", "9а", "Школа"]:
    print(f"  {вершина}: {' / '.join(путь_к_корню(родители, вершина))}")

Цикл поднимается по родителям, пока они есть. У корня родителя нет,
поэтому проверка `вершина in родители` останавливает подъём.
Развернув список, получаем путь сверху вниз — ровно как в адресе файла.

## Пробуем сами

### Задача 1. Количество листьев

По дереву, заданному словарём «вершина → список потомков»,
верните количество листьев.

In [ ]:
def листьев(дерево):
    return ...

In [ ]:
si.check("1", листьев, [
    ({"А": ["Б", "В"], "Б": [], "В": []}, 2),
    ({"А": []}, 1),
    ({"А": ["Б"], "Б": ["В"], "В": []}, 1),
])

### Задача 2. Высота дерева

Верните высоту дерева — наибольший уровень среди всех вершин.
У дерева из одной вершины высота 0.

Используйте обход из примера 2.

In [ ]:
def высота(дерево, корень):
    return ...

In [ ]:
si.check("2", высота, [
    (({"А": ["Б", "В"], "Б": [], "В": []}, "А"), 1),
    (({"А": []}, "А"), 0),
    (({"А": ["Б"], "Б": ["В"], "В": []}, "А"), 2),
])

### Задача 3. Вершин в полном дереве

Сколько всего вершин в полном двоичном дереве высоты 5?

Вспомните формулу из теории и впишите ответ числом.

In [ ]:
ответ = 0

si.check_value("3", ответ, "da4ea2a5506f2693",
               hint="Уровни дают 1 + 2 + 4 + 8 + 16 + 32 вершин.")

## Домашнее задание

### Домашнее задание 1. Полный путь к вершине

По словарю «потомок → родитель» и имени вершины верните полный путь
от корня, соединённый символом `/`.

`путь(родители, "1а")` → `"Школа/Началка/1а"`

In [ ]:
def путь(родители, вершина):
    return ...

In [ ]:
si.check("дз1", путь, [
    (({"Началка": "Школа", "1а": "Началка"}, "1а"), "Школа/Началка/1а"),
    (({"Началка": "Школа"}, "Школа"), "Школа"),
    (({"б": "а", "в": "б", "г": "в"}, "г"), "а/б/в/г"),
])

### Домашнее задание 2. Размер поддерева

Верните количество вершин в поддереве, растущем из заданной вершины,
включая её саму.

Обходите так же, как в примере 2, но начинайте не с корня.

In [ ]:
def размер_поддерева(дерево, вершина):
    return ...

In [ ]:
si.check("дз2", размер_поддерева, [
    (({"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}, "А"), 4),
    (({"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}, "Б"), 2),
    (({"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}, "В"), 1),
])

### Домашнее задание 3. Вершины полного дерева

Напишите функцию, которая по высоте полного двоичного дерева возвращает
список: сколько вершин на каждом уровне, начиная с корня.

`по_уровням(3)` → `[1, 2, 4, 8]`

А вторая функция пусть считает общее число вершин — но **не суммой
списка**, а по формуле из теории. Проверьте, что оба способа
дают одно и то же.

In [ ]:
def по_уровням(высота):
    return ...


def всего_вершин(высота):
    return ...

In [ ]:
si.check("дз3", по_уровням, [
    (3, [1, 2, 4, 8]),
    (0, [1]),
    (5, [1, 2, 4, 8, 16, 32]),
])

si.check("дз3б", всего_вершин, [
    (3, 15),
    (0, 1),
    (5, 63),
])

---

### Подумайте

В полном двоичном дереве листьев примерно столько же, сколько всех
остальных вершин вместе взятых. Проверьте это на дереве высоты 5:
листьев 32, остальных 31.

Получается, что **половина любого большого дерева — это его последний
уровень**. Именно поэтому алгоритмы, обходящие дерево целиком, тратят
основное время на листья, а алгоритмы, идущие от корня вниз по одной
ветви, работают так быстро.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 4](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-04.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 6 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-06.ipynb)